# Inference Notebook — ConvVAE

This notebook loads a saved checkpoint (`*.pt`) and runs the trained `ConvVAE` in inference mode on:
- a single image file, or
- a specified list of image files, or
- all PNG images in a directory.

It produces reconstruction images and saves a combined grid (top = real, bottom = recon) to the `out_dir` you specify.

Usage notes:
- You must provide model hyperparameters that match how the checkpoint was trained (at least `z_dim` and `image_size`).
- The notebook tries to find the project `src` folder and add it to `sys.path`. If you run this from a different working directory, update `repo_root` accordingly.
- Example outputs are saved in `out_dir` as `recon_<basename>.png` and a combined grid `recon_batch_###.png` for each processed batch.

In [ ]:
# Imports and repo/src path setup
import sys
from pathlib import Path
import os
from typing import List

import torch
import torchvision.transforms as T
from PIL import Image
import glob

# Find repository root by searching for a 'src' directory upwards from the current working directory
cwd = Path.cwd()
p = cwd
repo_root = None
for _ in range(6):
    if (p / 'src').exists():
        repo_root = p
        break
    if p.parent == p:
        break
    p = p.parent
if repo_root is None:
    # fallback: assume parent of current dir
    repo_root = cwd.parent
sys.path.insert(0, str(repo_root / 'src'))

from cpi_vae.model import ConvVAE
from cpi_vae.utils import save_reconstructions

print('repo_root =', repo_root)

In [ ]:
# Helper functions for loading images, model, and running inference
def get_image_paths(input_path: str) -> List[str]:
    p = Path(input_path)
    if p.is_dir():
        return sorted([str(x) for x in p.glob('*.png')])
    if p.is_file():
        return [str(p)]
    # maybe it's a glob or comma-separated list
    if '*' in input_path:
        return sorted(glob.glob(input_path))
    if ',' in input_path:
        return [s.strip() for s in input_path.split(',') if s.strip()]
    raise FileNotFoundError(f'No file or directory found at: {input_path}')

def load_images_tensor(paths: List[str], image_size: int, device: torch.device):
    transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.CenterCrop(image_size),
        T.ToTensor(),
    ])
    imgs = []
    for p in paths:
        im = Image.open(p).convert('L')
        t = transform(im)
        imgs.append(t)
    batch = torch.stack(imgs, dim=0).to(device)
    return batch

def load_model_from_checkpoint(ckpt_path: str, z_dim: int, image_size: int, device: torch.device):
    ckpt = torch.load(ckpt_path, map_location=device)
    model = ConvVAE(in_channels=1, z_dim=z_dim, image_size=image_size).to(device)
    # checkpoint originally saved {'epoch','model','opt'}
    if 'model' in ckpt:
        state = ckpt['model']
    else:
        state = ckpt
    model.load_state_dict(state)
    model.eval()
    return model

def run_inference_and_save(model: torch.nn.Module, input_paths: List[str], image_size: int, out_dir: str, device: torch.device, batch_size: int = 32):
    os.makedirs(out_dir, exist_ok=True)
    # process in batches
    for i in range(0, len(input_paths), batch_size):
        batch_paths = input_paths[i:i+batch_size]
        xb = load_images_tensor(batch_paths, image_size=image_size, device=device)
        with torch.no_grad():
            recon, mu, logvar = model(xb)
        # save combined grid for this batch
        out_grid = Path(out_dir) / f'recon_batch_{i//batch_size:03d}.png'
        # prepare CPU tensors for save_reconstructions (expects [N,C,H,W])
        save_reconstructions(xb.cpu(), recon.cpu(), str(out_grid))
        print('Saved grid:', out_grid)
        # also save individual reconstructions per image
        for j, p in enumerate(batch_paths):
            base = Path(p).stem
            out_single = Path(out_dir) / f'recon_{base}.png'
            # save single pair as a tiny grid of 1 (top real, bottom recon)
            save_reconstructions(xb.cpu()[j:j+1], recon.cpu()[j:j+1], str(out_single), n=1)
            print('Saved:', out_single)

In [ ]:
# Example: set parameters and run
checkpoint_path = 'checkpoints/run_20240101_000000/vae_epoch050.pt'  # <-- update to your checkpoint file
input_spec = 'data/example_images'   # can be: single file, directory, glob '*.png' or comma-separated list
out_dir = 'inference_outputs'
image_size = 224   # must match training image_size
z_dim = 128        # must match training z_dim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Loading model...')
model = load_model_from_checkpoint(checkpoint_path, z_dim=z_dim, image_size=image_size, device=device)
print('Finding input images...')
paths = []
try:
    paths = get_image_paths(input_spec)
except FileNotFoundError:
    # as a convenience, if `input_spec` is a literal path to a single file
    if Path(input_spec).exists():
        paths = [str(Path(input_spec))]
    else:
        raise

print(f'Found {len(paths)} images')
if len(paths) == 0:
    raise RuntimeError('No images found; update `input_spec` to point to a file or directory')

run_inference_and_save(model, paths, image_size=image_size, out_dir=out_dir, device=device)

Notes and troubleshooting:
- Ensure `image_size` and `z_dim` match the values used during training; otherwise the state dict won't load or reconstructions will be invalid.
- If the checkpoint contains a different key layout, inspect the checkpoint with `torch.load(checkpoint_path)` and adjust `load_model_from_checkpoint` accordingly.
- To quickly test the pipeline without GPU, set `device = torch.device('cpu')` and run with a small `--max_samples` or a small directory.
- If you want to visualize inline in the notebook instead of saving to disk, you can display the saved PNG using `from IPython.display import Image, display` and `display(Image(str(out_grid)))`.

If you want, I can add an inline display cell that shows the first batch's grid inside the notebook automatically.